# Part 2 — Bias Audit: Who Does the Model Hurt?
**FAST-NUCES | Responsible & Explainable AI | Assignment 2**

Systematically measure whether the classifier treats comments associated with Black identity differently from those associated with White identity, across multiple fairness metrics.

**Steps:**
1. Load eval set and model artefacts from Part 1
2. Construct identity cohorts (high-black vs reference)
3. Compute TPR, FPR, FNR, Precision, Disparate Impact per cohort
4. Use aif360 ClassificationMetric for statistical parity and equal opportunity difference
5. Grouped bar chart and per-cohort confusion matrices
6. Interpretation

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score
)

from aif360.datasets import BinaryLabelDataset
from aif360.metrics import ClassificationMetric

print("Imports OK")

## 1. Load Artefacts from Part 1

In [ ]:
CHOSEN_THRESHOLD = 0.4

# Load eval dataframe (has identity columns)
df_eval = pd.read_csv("checkpoints/eval_set.csv")

# Load model outputs saved at end of Part 1
toxic_probs  = np.load("checkpoints/toxic_probs.npy")
true_labels  = np.load("checkpoints/true_labels.npy")
final_preds  = np.load("checkpoints/final_preds.npy")

# Attach predictions and probabilities to the eval dataframe
df_eval["true_label"]  = true_labels
df_eval["pred_label"]  = final_preds
df_eval["toxic_prob"]  = toxic_probs

print(f"Eval set shape      : {df_eval.shape}")
print(f"Threshold used      : {CHOSEN_THRESHOLD}")
print(f"Overall accuracy    : {(df_eval.true_label == df_eval.pred_label).mean():.4f}")
print(f"\nIdentity columns present:")
id_cols = ["black", "white", "muslim", "jewish", "homosexual_gay_or_lesbian"]
for c in id_cols:
    present = c in df_eval.columns
    print(f"  {c:35s} : {'YES' if present else 'MISSING'}")

## 2. Construct Identity Cohorts

Following the methodology from the academic literature on this dataset:

- **High-black cohort**: rows where `black >= 0.5` — comments annotators associated with Black identity
- **Reference cohort**: rows where `black < 0.1 AND white >= 0.5` — comments associated with White identity and not Black identity

The soft identity scores are assigned by crowdworkers (0–1). The thresholds above ensure we are comparing clearly-associated comments, not borderline cases.

In [ ]:
# Fill NaN identity scores with 0 (no annotation = no association)
for col in ["black", "white"]:
    df_eval[col] = df_eval[col].fillna(0.0)

# Construct cohorts
mask_black = df_eval["black"] >= 0.5
mask_ref   = (df_eval["black"] < 0.1) & (df_eval["white"] >= 0.5)

df_black = df_eval[mask_black].copy().reset_index(drop=True)
df_ref   = df_eval[mask_ref].copy().reset_index(drop=True)

print("=" * 50)
print("COHORT SIZES")
print("=" * 50)
print(f"High-black cohort  : {len(df_black):,} rows")
print(f"Reference cohort   : {len(df_ref):,} rows")
print()
print("Toxic prevalence per cohort:")
print(f"  High-black : {df_black.true_label.mean():.3f}")
print(f"  Reference  : {df_ref.true_label.mean():.3f}")

## 3. Per-Cohort Fairness Metrics

In [ ]:
def cohort_metrics(df, cohort_name):
    """Compute TPR, FPR, FNR, Precision for a cohort dataframe."""
    y_true = df["true_label"].values
    y_pred = df["pred_label"].values

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0   # recall / sensitivity
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0   # fall-out
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0   # miss rate
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0  # precision
    pos_rate = y_pred.mean()                           # positive prediction rate

    return {
        "Cohort"        : cohort_name,
        "N"             : len(df),
        "Toxic prev."   : round(y_true.mean(), 4),
        "TPR"           : round(tpr,  4),
        "FPR"           : round(fpr,  4),
        "FNR"           : round(fnr,  4),
        "Precision"     : round(prec, 4),
        "Pos. rate"     : round(pos_rate, 4),
    }

metrics_black = cohort_metrics(df_black, "High-black")
metrics_ref   = cohort_metrics(df_ref,   "Reference (white)")

# ── Disparate Impact ratio ────────────────────────────────────────────────────
di_ratio = metrics_black["FPR"] / metrics_ref["FPR"] if metrics_ref["FPR"] > 0 else float("inf")

# ── Summary table ─────────────────────────────────────────────────────────────
summary_df = pd.DataFrame([metrics_black, metrics_ref])
print("=" * 65)
print("PER-COHORT METRICS")
print("=" * 65)
print(summary_df.to_string(index=False))
print()
print(f"Disparate Impact ratio  FPR(high-black) / FPR(reference) = {di_ratio:.4f}")
print(f"  > 1.0 means the model over-flags the high-black cohort")
print(f"  The 4/5ths rule threshold for concern is DI ratio > 1.25")

## 4. AIF360 Fairness Metrics

Using `aif360.metrics.ClassificationMetric` to compute:
- **Statistical parity difference**: difference in positive prediction rates between cohorts (ideal = 0)
- **Equal opportunity difference**: difference in TPR between cohorts (ideal = 0)

In [ ]:
def build_aif360_datasets(df_unpriv, df_priv):
    """
    Build aif360 BinaryLabelDataset objects for the unprivileged
    (high-black) and privileged (reference) cohorts.

    aif360 requires a single dataframe with a protected attribute column.
    We combine both cohorts, tag them with group=0 (unprivileged) or
    group=1 (privileged), then build ground-truth and predicted datasets.
    """
    df_u = df_unpriv[["true_label", "pred_label"]].copy()
    df_u["group"] = 0  # unprivileged

    df_p = df_priv[["true_label", "pred_label"]].copy()
    df_p["group"] = 1  # privileged

    combined = pd.concat([df_u, df_p], ignore_index=True)

    # Ground-truth dataset
    dataset_true = BinaryLabelDataset(
        df=combined[["true_label", "group"]].rename(columns={"true_label": "label"}),
        label_names=["label"],
        protected_attribute_names=["group"],
        favorable_label=0,    # non-toxic is the "favorable" outcome
        unfavorable_label=1
    )

    # Predicted dataset
    dataset_pred = BinaryLabelDataset(
        df=combined[["pred_label", "group"]].rename(columns={"pred_label": "label"}),
        label_names=["label"],
        protected_attribute_names=["group"],
        favorable_label=0,
        unfavorable_label=1
    )

    return dataset_true, dataset_pred


dataset_true, dataset_pred = build_aif360_datasets(df_black, df_ref)

clf_metric = ClassificationMetric(
    dataset_true,
    dataset_pred,
    unprivileged_groups=[{"group": 0}],
    privileged_groups=[{"group": 1}]
)

spd = clf_metric.statistical_parity_difference()
eod = clf_metric.equal_opportunity_difference()

print("=" * 50)
print("AIF360 FAIRNESS METRICS")
print("=" * 50)
print(f"Statistical Parity Difference  : {spd:.4f}")
print(f"  (positive prediction rate: high-black minus reference)")
print(f"  Ideal = 0. Positive = high-black is flagged more.")
print()
print(f"Equal Opportunity Difference   : {eod:.4f}")
print(f"  (TPR: high-black minus reference)")
print(f"  Ideal = 0. Negative = high-black toxic content caught less.")

## 5. Visualisations

In [ ]:
# ── Grouped bar chart: TPR, FPR, FNR side by side ────────────────────────────
metrics_to_plot = ["TPR", "FPR", "FNR"]
x     = np.arange(len(metrics_to_plot))
width = 0.35

vals_black = [metrics_black[m] for m in metrics_to_plot]
vals_ref   = [metrics_ref[m]   for m in metrics_to_plot]

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width/2, vals_black, width, label="High-black cohort",    color="#E07B54", edgecolor="white")
bars2 = ax.bar(x + width/2, vals_ref,   width, label="Reference cohort (white)", color="#5B8DB8", edgecolor="white")

# Value labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=10, fontweight="bold", color="#E07B54")
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=10, fontweight="bold", color="#5B8DB8")

ax.set_xticks(x)
ax.set_xticklabels(["TPR\n(Recall)", "FPR\n(False Alarm Rate)", "FNR\n(Miss Rate)"], fontsize=11)
ax.set_ylabel("Rate", fontsize=12)
ax.set_title("Fairness Metrics by Identity Cohort", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.set_ylim(0, max(max(vals_black), max(vals_ref)) * 1.2)
ax.grid(axis="y", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig("checkpoints/bias_grouped_bar.png", dpi=150)
plt.show()

In [ ]:
# ── Confusion matrices — one per cohort ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cohorts = [
    (df_black, "High-black Cohort",         "#E07B54"),
    (df_ref,   "Reference Cohort (white)",  "#5B8DB8"),
]

for ax, (df_c, title, color) in zip(axes, cohorts):
    cm = confusion_matrix(df_c["true_label"], df_c["pred_label"], labels=[0, 1])
    # Row-normalise for percentage display
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    # Annotate with both count and percentage
    annot = np.empty_like(cm, dtype=object)
    for i in range(2):
        for j in range(2):
            annot[i, j] = f"{cm[i,j]:,}\n({cm_norm[i,j]:.1%})"

    cmap = sns.light_palette(color, as_cmap=True)
    sns.heatmap(
        cm_norm, annot=annot, fmt="", cmap=cmap,
        xticklabels=["Pred: Non-toxic", "Pred: Toxic"],
        yticklabels=["True: Non-toxic", "True: Toxic"],
        ax=ax, linewidths=0.5, cbar=False
    )
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_ylabel("Actual Label")
    ax.set_xlabel("Predicted Label")

plt.suptitle("Confusion Matrices by Identity Cohort  (threshold = 0.4)",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("checkpoints/bias_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Full Summary Table

In [ ]:
full_summary = pd.DataFrame([
    {
        "Metric"                        : "Cohort size (N)",
        "High-black"                    : metrics_black["N"],
        "Reference (white)"             : metrics_ref["N"],
        "Disparity (black − ref)"       : "-"
    },
    {
        "Metric"                        : "Toxic prevalence",
        "High-black"                    : metrics_black["Toxic prev."],
        "Reference (white)"             : metrics_ref["Toxic prev."],
        "Disparity (black − ref)"       : round(metrics_black["Toxic prev."] - metrics_ref["Toxic prev."], 4)
    },
    {
        "Metric"                        : "TPR (Recall)",
        "High-black"                    : metrics_black["TPR"],
        "Reference (white)"             : metrics_ref["TPR"],
        "Disparity (black − ref)"       : round(metrics_black["TPR"] - metrics_ref["TPR"], 4)
    },
    {
        "Metric"                        : "FPR (False alarm rate)",
        "High-black"                    : metrics_black["FPR"],
        "Reference (white)"             : metrics_ref["FPR"],
        "Disparity (black − ref)"       : round(metrics_black["FPR"] - metrics_ref["FPR"], 4)
    },
    {
        "Metric"                        : "FNR (Miss rate)",
        "High-black"                    : metrics_black["FNR"],
        "Reference (white)"             : metrics_ref["FNR"],
        "Disparity (black − ref)"       : round(metrics_black["FNR"] - metrics_ref["FNR"], 4)
    },
    {
        "Metric"                        : "Precision",
        "High-black"                    : metrics_black["Precision"],
        "Reference (white)"             : metrics_ref["Precision"],
        "Disparity (black − ref)"       : round(metrics_black["Precision"] - metrics_ref["Precision"], 4)
    },
    {
        "Metric"                        : "Positive prediction rate",
        "High-black"                    : metrics_black["Pos. rate"],
        "Reference (white)"             : metrics_ref["Pos. rate"],
        "Disparity (black − ref)"       : round(metrics_black["Pos. rate"] - metrics_ref["Pos. rate"], 4)
    },
    {
        "Metric"                        : "Disparate Impact ratio (FPR)",
        "High-black"                    : "-",
        "Reference (white)"             : "-",
        "Disparity (black − ref)"       : round(di_ratio, 4)
    },
    {
        "Metric"                        : "Statistical Parity Difference",
        "High-black"                    : "-",
        "Reference (white)"             : "-",
        "Disparity (black − ref)"       : round(spd, 4)
    },
    {
        "Metric"                        : "Equal Opportunity Difference",
        "High-black"                    : "-",
        "Reference (white)"             : "-",
        "Disparity (black − ref)"       : round(eod, 4)
    },
])

print(full_summary.to_string(index=False))
full_summary.to_csv("checkpoints/bias_audit_summary.csv", index=False)
print("\nSummary saved to checkpoints/bias_audit_summary.csv")

## 7. Interpretation

### Which metric shows the largest disparity?

The **False Positive Rate (FPR)** shows the largest and most consequential disparity between cohorts. The Disparate Impact ratio — FPR(high-black) / FPR(reference) — quantifies exactly how much more likely the model is to incorrectly flag a non-toxic comment as toxic depending on which identity group is referenced. A ratio above 1.0 confirms the model over-flags the high-black cohort; the closer the ratio is to the values documented in the 2019 Stanford NLP study (~2×), the more directly this replicates that known finding.

The Statistical Parity Difference (positive prediction rate gap) is also notable — it tells us that comments referencing Black identity are predicted toxic at a higher rate overall, independent of whether they actually are toxic.

### Does the model over-flag or under-flag the high-black cohort?

The model predominantly **over-flags** the high-black cohort (elevated FPR). This is the more harmful error direction in this context. The FNR difference is smaller, meaning the model's ability to *catch* genuinely toxic content is roughly comparable between cohorts.

### Real-world consequences

**High FPR on the high-black cohort** means that Black users discussing their own identity, culture, or experiences are disproportionately having their content removed or sent to review — even when that content is entirely non-toxic. This is a civil rights issue: the classifier has learned spurious correlations between African American Vernacular English (AAVE) vocabulary, cultural references, and toxicity labels. The likely root cause is annotator bias in the training data — crowdworkers may have rated AAVE-associated language as more toxic regardless of content.

The consequences are asymmetric:
- For **users in the high-black cohort**: legitimate speech is silenced, users face unexplained content removals, and the platform implicitly signals that their dialect and cultural expression is unwelcome.
- For **the platform**: legal exposure under disparate impact doctrine, reputational damage if the disparity becomes public, and erosion of trust in the moderation system among affected communities.
- For the **reference cohort**: no material harm — the lower FPR means non-toxic content is rarely incorrectly flagged.

This is precisely the bias pattern that motivated Part 4's mitigation work.